# Oxygen sampling–mapping trend differences

Reproduces the two-period figure with Olivelli et al. (2026). Column 1 covers 1965–2021 and column 2 covers 1993–2021. The notebook saves both the original overlapping-distribution design and a box-and-whisker design to .


In [1]:
"""Plot global oxygen sampling/mapping trend differences for two periods.

Column 1 is the full 1965--2021 record and column 2 retains 1993--2021.
Two PDF variants are written: the original overlaid half-violin design and a
horizontal box-and-whisker design showing every product plus the combined set.
"""

import os
import pickle
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-db9274")

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter
import numpy as np


SOURCE_DIR = Path("/scratch/gpfs/LRGROUP/db9274/Variability_trends_EC/Paper_figures")
CACHE_FILE = SOURCE_DIR / (
    "cache_sampling_mapping_trends_different_periods/"
    "global_trend_sampling_mapping_samples_full_upper_below_1965_2021_"
    "6_products_olivelli_v3.pkl"
)
OUTPUT_DIR = Path.cwd() / "Figures"
VIOLIN_OUTPUT = OUTPUT_DIR / "oxygen_sampling_mapping_difference_trends_two_periods.pdf"
BOX_OUTPUT = OUTPUT_DIR / "oxygen_sampling_mapping_difference_trends_two_periods_boxplots.pdf"

PERIODS = ["1965–2021", "1993–2021"]
DEPTHS = ["Full water column", "Upper 2000 m", "Below 2000 m"]
PRODUCTS = [
    "observation-Gouretski(2024)",
    "observation-Ito(2024; NN)",
    "observation-Ito(2024; RF)",
    "observation-Ito(2022; 5 year)",
    "observation-Roach and Bindoff",
    "observation-Olivelli(2026)",
]
LABELS = {
    "observation-Gouretski(2024)": "Gouretski et al. (2024)",
    "observation-Ito(2024; NN)": "Ito et al. (2024): Neural network",
    "observation-Ito(2024; RF)": "Ito et al. (2024): Random forest",
    "observation-Ito(2022; 5 year)": "Ito (2022)",
    "observation-Roach and Bindoff": "Roach & Bindoff (2023)",
    "observation-Olivelli(2026)": "Olivelli et al. (2026)",
}
COLORS = dict(zip(PRODUCTS, ["blue", "green", "red", "purple", "#D55E00", "#7B2CBF"]))
XLABEL = "Sampling–mapping difference ($\\mu$mol kg$^{-1}$ decade$^{-1}$)"

plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "figure.dpi": 130,
    "savefig.dpi": 300,
})


def gaussian_pdf(values, grid):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    std = np.std(values, ddof=1)
    bandwidth = 1.06 * std * values.size ** (-1 / 5)
    if not np.isfinite(bandwidth) or bandwidth == 0:
        bandwidth = max(np.ptp(values), 1.0) / 20
    scaled = (grid[:, None] - values[None, :]) / bandwidth
    return np.exp(-0.5 * scaled**2).sum(axis=1) / (
        values.size * bandwidth * np.sqrt(2 * np.pi)
    )


def symmetric_xlim(samples):
    selected = samples[samples["period"].isin(PERIODS)]
    values = selected["sampling_mapping_difference"].to_numpy(float)
    values = values[np.isfinite(values)]
    lo, hi = np.percentile(values, [1, 99])
    extent = 1.2 * max(abs(lo), abs(hi))
    return -extent, extent


def add_percent_axis(ax, data):
    full = data.drop_duplicates(["model", "ensemble"])["fully_sampled_trend"].to_numpy(float)
    reference = abs(np.nanmedian(full))
    if np.isfinite(reference) and reference > 0:
        sec = ax.secondary_xaxis(
            "top",
            functions=(lambda x: 100 * x / reference, lambda pct: pct * reference / 100),
        )
        sec.xaxis.set_major_formatter(FuncFormatter(lambda value, pos: f"{value:g}%"))
        sec.tick_params(axis="x", labelsize=8, pad=1.5)


def panel_data(samples, period, depth):
    return samples[(samples["period"] == period) & (samples["depth"] == depth)]


def plot_half_violin(ax, grid, values, side, color, alpha, linewidth):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    density = gaussian_pdf(values, grid)
    width = side * 0.38 * density / np.nanmax(density)
    ax.fill_between(grid, 0, width, color=color, alpha=alpha, linewidth=0)
    ax.plot(grid, width, color=color, lw=linewidth)
    median = np.median(values)
    ax.plot([median, median], [0, side * 0.35], color=color, lw=1.5)


def format_common_panel(ax, xlim, data, bottom):
    ax.axvline(0, color="0.35", lw=0.9, ls="--")
    ax.set_xlim(xlim)
    ax.grid(axis="x", ls=":", alpha=0.25)
    ax.tick_params(axis="both", labelsize=8.5, pad=1.5)
    ax.set_xlabel(XLABEL if bottom else "")
    add_percent_axis(ax, data)


def make_violin_figure(samples, xlim):
    fig, axes = plt.subplots(3, 2, figsize=(8, 5.5), squeeze=False)
    fig.subplots_adjust(left=0.12, right=0.98, top=0.91, bottom=0.17, wspace=0.16, hspace=0.42)
    grid = np.linspace(*xlim, 300)
    panel = 0
    for row, depth in enumerate(DEPTHS):
        for col, period in enumerate(PERIODS):
            ax = axes[row, col]
            data = panel_data(samples, period, depth)
            plot_half_violin(ax, grid, data["sampling_mapping_difference"], 1, "black", 0.18, 1.9)
            for product in PRODUCTS:
                values = data.loc[data["product"] == product, "sampling_mapping_difference"]
                plot_half_violin(ax, grid, values, -1, COLORS[product], 0.24, 1.7)
            ax.axhline(0, color="0.25", lw=1)
            ax.set_ylim(-0.5, 0.5)
            ax.set_yticks([])
            format_common_panel(ax, xlim, data, row == 2)
            if row == 0:
                ax.set_title(period, pad=24)
            if col == 0:
                ax.set_ylabel(depth, fontsize=9.5, fontweight="bold", labelpad=22)
            ax.text(0.01, 0.98, chr(ord("a") + panel), transform=ax.transAxes,
                    ha="left", va="top", fontsize=11, fontweight="bold")
            panel += 1
    handles = [Patch(facecolor="black", alpha=0.18, label="Combined estimate")]
    handles += [Patch(facecolor=COLORS[p], alpha=0.35, label=LABELS[p]) for p in PRODUCTS]
    fig.legend(handles=handles, loc="lower center", ncol=4, frameon=False, fontsize=7.7,
               bbox_to_anchor=(0.55, -0.005), columnspacing=0.8, handlelength=1.1)
    fig.savefig(VIOLIN_OUTPUT, bbox_inches="tight")
    plt.close(fig)


def make_box_figure(samples, xlim):
    fig, axes = plt.subplots(3, 2, figsize=(10.5, 9.0), squeeze=False)
    fig.subplots_adjust(left=0.12, right=0.98, top=0.94, bottom=0.15, wspace=0.16, hspace=0.32)
    box_labels = ["Combined estimate"] + [LABELS[p] for p in PRODUCTS]
    box_colors = ["black"] + [COLORS[p] for p in PRODUCTS]
    positions = np.arange(len(box_labels), 0, -1)
    panel = 0
    for row, depth in enumerate(DEPTHS):
        for col, period in enumerate(PERIODS):
            ax = axes[row, col]
            data = panel_data(samples, period, depth)
            groups = [data["sampling_mapping_difference"].dropna().to_numpy()]
            groups += [data.loc[data["product"] == p, "sampling_mapping_difference"].dropna().to_numpy()
                       for p in PRODUCTS]
            result = ax.boxplot(groups, vert=False, positions=positions, widths=0.58,
                                patch_artist=True, showfliers=True,
                                medianprops={"color": "white", "linewidth": 1.4},
                                whiskerprops={"linewidth": 1}, capprops={"linewidth": 1},
                                flierprops={"marker": "o", "markersize": 2.8, "alpha": 0.55})
            for patch, color in zip(result["boxes"], box_colors):
                patch.set(facecolor=color, edgecolor=color, alpha=0.5)
            ax.set_yticks(positions)
            ax.set_yticklabels([])
            ax.set_ylim(0.35, len(box_labels) + 0.65)
            format_common_panel(ax, xlim, data, row == 2)
            if row == 0:
                ax.set_title(period, pad=24)
            if col == 0:
                ax.text(-0.14, 0.5, depth, transform=ax.transAxes, rotation=90,
                        va="center", ha="center", fontsize=9.5, fontweight="bold")
            ax.text(0.01, 0.98, chr(ord("a") + panel), transform=ax.transAxes,
                    ha="left", va="top", fontsize=11, fontweight="bold")
            panel += 1
    handles = [Patch(facecolor="black", edgecolor="black", alpha=0.5, label="Combined estimate")]
    handles += [Patch(facecolor=COLORS[p], edgecolor=COLORS[p], alpha=0.5, label=LABELS[p])
                for p in PRODUCTS]
    fig.legend(handles=handles, loc="lower center", ncol=4, frameon=False, fontsize=8.0,
               bbox_to_anchor=(0.55, 0.005), columnspacing=0.9, handlelength=1.5)
    fig.savefig(BOX_OUTPUT, bbox_inches="tight")
    plt.close(fig)


def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    with CACHE_FILE.open("rb") as handle:
        samples = pickle.load(handle)
    missing_products = set(PRODUCTS) - set(samples["product"])
    missing_periods = set(PERIODS) - set(samples["period"])
    if missing_products or missing_periods:
        raise ValueError(f"Cache missing products={missing_products}, periods={missing_periods}")
    xlim = symmetric_xlim(samples)
    make_violin_figure(samples, xlim)
    make_box_figure(samples, xlim)
    print(f"Saved {VIOLIN_OUTPUT}")
    print(f"Saved {BOX_OUTPUT}")


if __name__ == "__main__":
    main()


Saved /scratch/gpfs/LRGROUP/db9274/Variability_trends_EC/Paper_figures_p2/Figures/oxygen_sampling_mapping_difference_trends_two_periods.pdf
Saved /scratch/gpfs/LRGROUP/db9274/Variability_trends_EC/Paper_figures_p2/Figures/oxygen_sampling_mapping_difference_trends_two_periods_boxplots.pdf
